In [1]:
!pip install  scikit-learn

In [2]:
# Импортируем библиотеки для работы с датасетом
import warnings, time
import numpy as np, pandas as pd

# Импортируем необходимые инструменты из sklearn
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.exceptions import ConvergenceWarning

In [7]:
# Записываем датасет в переменную, определяем таргет и задаем сид
CSV = "https://raw.githubusercontent.com/Khamoon7/GeoATM-popularity/refs/heads/main/data/train_data.csv"; TARGET = "target"; SEED = 42

# Признаки, которые не будут участвовать в обучении
DROP = ["id","atm_group","address_raw","address_geocoded","geo_lon","geo_lat","municipality","city","street","house"]

# Бинарные признаки - приведём к 0/1
BIN  = ["is_24_7","contactless_tech","qr_codes","usd_available","eur_available",
        "cash_in","cash_out","cashless_pay","account_statement","access_for_disabled",
        "transfer_p2p","transfer_a2a","loan_payments","has_subway_nearby"]

# Отключаем предупреждения от OneHotEncoder и Lasso
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn.preprocessing._encoders")
warnings.filterwarnings("ignore", category=ConvergenceWarning)


# ------- Подготовка датасета -------

# Загружаем датасет
df = pd.read_csv(CSV)

# Удаляем строки без target
df.dropna(subset=[TARGET], inplace=True)

# Удаляем ненужные столбцы
df.drop(columns=[c for c in DROP if c in df.columns], errors="ignore", inplace=True)

display(df)

,country,region,population_density_per_km2,target,is_24_7,contactless_tech,qr_codes,usd_available,eur_available,cash_in,...,nearest_public_transport_dist_m,count_public_transport_300m,nearest_parking_dist_m,count_parking_300m,nearest_education_dist_m,count_education_300m,nearest_subway_dist_m,nearest_post_offices_dist_m,count_post_offices_300m,has_subway_nearby
0,Россия,Республика Калмыкия,3.52,0.019958,False,False,False,False,False,True,...,93.6,5,143.7,3,247.3,1,0.0,0.0,0,False
1,Россия,Ульяновская область,32.09,-0.089111,False,True,False,False,False,True,...,89.9,6,NaN,0,260.8,1,0.0,220.4,2,False
2,Россия,Хабаровский край,1.60,-0.048769,False,True,True,False,False,True,...,33.8,8,112.9,6,0.0,0,0.0,186.6,2,False
3,Россия,Республика Саха (Якутия),0.31,-0.041545,False,False,False,False,False,True,...,119.8,7,246.9,4,195.4,5,0.0,167.2,1,False
4,Россия,Забайкальский край,2.28,0.025479,False,False,False,False,False,True,...,70.6,3,48.3,5,NaN,0,NaN,NaN,0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6195,Россия,Республика Калмыкия,3.52,-0.097043,False,False,False,False,False,False,...,192.4,1,631.5,0,NaN,0,NaN,NaN,0,False
6196,Россия,Республика Калмыкия,3.52,-0.090848,False,False,False,False,False,False,...,NaN,0,780.6,0,NaN,0,NaN,NaN,0,False
6197,Россия,Республика Калмыкия,3.52,0.036916,False,False,False,False,False,False,...,NaN,0,NaN,0,NaN,0,NaN,NaN,0,False
6198,Россия,Республика Калмыкия,3.52,-0.135684,True,False,False,False,False,False,...,131.8,2,144.1,11,112.2,2,0.0,0.0,0,False


In [14]:
# Преобразования бинарных признаков в 0/1
present_bin = [c for c in BIN if c in df.columns]

for c in present_bin:
    df[c] = df[c].astype(float)   # True → 1.0, False → 0.0

# Работа с категориальным признаком region
cat_cols = []
# Считаем частоту регионов
cnt = df["region"].value_counts(dropna=False)
# Выделяем редкие регионы (<1% наблюдений)
rare = cnt[cnt < 0.01*len(df)].index
# Группируем редкие в категорию "__OTHER__"
df["region_grouped"] = df["region"].where(~df["region"].isin(rare), "__OTHER__")
cat_cols = ["region_grouped"]

# Создаем список числовых признаков (все числовые, кроме target и бинарных)
num_cols = (
    df.select_dtypes(include=[np.number]).columns
        .drop([TARGET], errors="ignore")
        .difference(present_bin)
        .tolist()
)

# Разделяем на X и y
X = df.drop(columns=[TARGET])
y = df[TARGET].values

# Настраиваем OHE — для дальнейшей кодировки регионов
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

# Преобразуем колонки, в которых есть NA
pre = ColumnTransformer([
    # Числовые: скейлим - медианой 
    ("num", Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc", StandardScaler())
    ]), num_cols),

    # Бинарные просто заполняем самым частым значением
    ("bin", Pipeline([
        ("imp", SimpleImputer(strategy="most_frequent"))
    ]), present_bin),

    # Категориальные заполняем самым частым значением
    ("cat", Pipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("ohe", ohe)
    ]), cat_cols),

], remainder="drop", verbose_feature_names_out=False)


# Разделяем на тестовую и тренировочную выборки
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=SEED)


# ------- Обучаем + подбираем гиперпараметры -------

# Собираем цепочку шагов
pipe = Pipeline([
    ("prep", pre),
    ("mdl", Lasso(random_state=SEED, max_iter=200_000))
])

# Подбор гиперпараметров
param_grid = {
    "mdl__alpha": np.logspace(-5, 2, 30),
    "mdl__tol": [1e-3, 3e-4],               
    "mdl__selection": ["cyclic"],       
}

# Запускаем GridSearchCV - подборка всех комбинаций параметров
t0 = time.time()
gcv = GridSearchCV(
    pipe,
    param_grid,
    scoring="neg_mean_squared_error",
    cv=3,
    n_jobs=-1,
    refit=True,
    verbose=1
)
gcv.fit(Xtr, ytr)
runtime = round(time.time() - t0, 2)

# Извлекаем лучшую модель
best = gcv.best_estimator_

# Записываем предсказания в переменную
y_tr = best.predict(Xtr)
y_te = best.predict(Xte)

# Считаем метрики
train_mse = float(mean_squared_error(ytr, y_tr))
test_rmse = float(np.sqrt(mean_squared_error(yte, y_te)))
test_r2   = float(r2_score(yte, y_te))

# Достаём фичи + коэффициенты
feat = best.named_steps["prep"].get_feature_names_out()
coef = best.named_steps["mdl"].coef_

# Кол-во ненулевых коэффициентов (важно для Lasso)
nz = int(np.count_nonzero(coef))

# Сохраняем коэффициенты
(
    pd.DataFrame({"feature": feat, "coef": coef})
      .sort_values(by="coef", key=lambda s: s.abs(), ascending=False)
      .to_csv("lasso_coefficients_grid_widealpha.csv", index=False)
)

# Сохраняем метрики
(
    pd.DataFrame([{
        "Model":"Lasso_grid_widealpha",
        "Train_MSE":train_mse,
        "Test_RMSE":test_rmse,
        "Test_R2":test_r2,
        **{k.replace("mdl__",""): v for k,v in gcv.best_params_.items()},
        "Nonzero": nz,
        "Runtime_sec": runtime,
        "Used_num": len(num_cols),
        "Used_bin": len(present_bin),
        "Used_cat_region": int(len(cat_cols)>0)
    }])
    .to_csv("lasso_metrics_grid_widealpha.csv", index=False)
)

# Выводим финальный результат
print("\nЛучшие параметры:", gcv.best_params_)
print(f"Train MSE: {train_mse:.6f} | Test RMSE: {test_rmse:.6f} | Test R^2: {test_r2:.6f} | Время работы: {runtime}s")

Fitting 3 folds for each of 60 candidates, totalling 180 fits

Лучшие параметры: {'mdl__alpha': np.float64(5.2983169062837125e-05), 'mdl__selection': 'cyclic', 'mdl__tol': 0.001}
Train MSE: 0.004504 | Test RMSE: 0.070724 | Test R^2: 0.345729 | Время работы: 4.41s


# Отчёт по модели Lasso Regression

## 1. Подготовка данных
- Удалены строки с пропущенным `target`.
- Исключены нерелевантные признаки:  
  `id, atm_group, address_raw, address_geocoded, geo_lon, geo_lat, municipality, city, street, house`
- Бинарные признаки приведены в формат `0/1`.
- Признак `region` обработан: редкие категории (<1% выборки) заменены на `__OTHER__`, далее применён OneHotEncoding.
- Разделение выборки: `train_test_split(test_size=0.25, random_state=42)`.

### Преобразование признаков
- Числовые признаки: Заполняем NA медианой
- Бинарные признаки: Заполняем NA часто встречаемым значением
- Категориальные признаки: Заполняем NA часто встречаемым значением

---

## 2. Модель и подбор гиперпараметров

### Модель
Использована **Lasso Regression** — линейная модель с L1-регуляризацией.

### Подбор гиперпараметров
Метод: **GridSearchCV**, CV = 3.  
Основная метрика: **Negative MSE**.

Сетка параметров:
```
alpha ∈ logspace(-5, 2, 30)   # 0.00001 → 100
tol ∈ [1e-3, 3e-4]
selection = 'cyclic'
```

---

## 3. Лучшие найденные параметры модели

```
{
  "alpha": 5.298316906283712e-05,
  "tol": 0.001,
  "selection": "cyclic"
}
```

---

## 4. Результаты качества модели

| Метрика | Значение |
|--------|----------|
| **Train MSE** | 0.004504 |
| **Test RMSE** | 0.070724 |
| **Test R²** | 0.345729 |


---

## 5. Наиболее значимые признаки (по абсолютному значению коэффициентов)

Крупнейшие коэффициенты показывают значимость:
1. **Региональных категорий** (`region_grouped_*`)
2. **Возможностей банкомата** (transfer_a2a, usd_available, eur_available)
3. **Инфраструктура** (pharmacies, malls, cafés)
4. **Транспортной доступности** (subway_distance)

---

## 6. Основные выводы

- Lasso показывает **умеренное качество**.
- Наибольший вклад вносят **регион**, **инфраструктура**, **функциональные параметры банкомата**.
- Низкое оптимальное `alpha` говорит о том, что модель лучше работает с **минимальной регуляризацией**
